In [ ]:
"""
Извлечение self-correction векторов через TransformerLens.
Версия 2: два новых метода извлечения.

МЕТОД А — Comparative Activation Slicing (Before vs After):
  Для каждого маркера берём активацию НА САМОМ МАРКЕРЕ ("wait")
  и сравниваем genuine vs stylistic напрямую.
  Не нужен control_run — только один прогон на полный текст.

МЕТОД Б — Counterfactual Activation Addition (CAA):
  Для genuine случая берём полный текст с маркером (correction_run).
  Создаём counterfactual: заменяем маркер на "продолжение ошибки"
  (следующий наиболее вероятный токен, который НЕ является коррекцией).
  Вектор = активация(correction) - активация(counterfactual).
"""
!pip install transformer_lens torch numpy scikit-learn matplotlib seaborn -q

In [ ]:
# Полная зачистка
!pip uninstall -y torch torchvision torchaudio transformers accelerate einops bitsandbytes -y

# Очищаем кэш
!rm -rf ~/.cache/huggingface/hub/
!rm -rf ~/.cache/huggingface/modules/

# Устанавливаем PyTorch 2.1.2
!pip install torch==2.1.2 torchvision==0.16.2 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121

# Устанавливаем совместимые версии
!pip install transformers==4.36.0
!pip install accelerate==0.25.0
!pip install einops==0.7.0

# РАБОЧАЯ УСТАНОВКА bitsandbytes для Kaggle
!pip install bitsandbytes==0.41.1 --no-deps
!pip install scipy

# Проверка
import torch
import transformers
import bitsandbytes

print(f"PyTorch version: {torch.__version__}")
print(f"Transformers version: {transformers.__version__}")
print(f"bitsandbytes version: {bitsandbytes.__version__}")

# Перезапуск ядра
import os
os.kill(os.getpid(), 9)

In [ ]:
!pip install numpy==1.23.5

!pip install torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118

!pip install transformers==4.35.0
!pip install accelerate==0.24.0
!pip install einops==0.6.1
!pip install scipy==1.10.1
!pip install bitsandbytes==0.41.0 --no-deps

import numpy as np
print(f"NumPy version: {np.__version__}")
print(f"NumPy path: {np.__file__}")

import torch
print(f"PyTorch version: {torch.__version__}")

import bitsandbytes
print(f"bitsandbytes version: {bitsandbytes.__version__}")

In [ ]:
# ШАГ 1: Полная зачистка
!pip uninstall bitsandbytes -y
!rm -rf bitsandbytes
!rm -rf ~/.cache/huggingface/modules/bitsandbytes*

# ШАГ 2: Клонируем репозиторий
!git clone https://github.com/TimDettmers/bitsandbytes.git
%cd bitsandbytes

# ШАГ 3: Компилируем для CUDA 12.8 (ваша версия)
!CUDA_VERSION=128 make cuda12x
!python setup.py install

# ШАГ 4: Возвращаемся в исходную директорию
%cd ..

# ШАГ 5: Проверка
import bitsandbytes
print(f"✅ bitsandbytes успешно скомпилирован и импортирован!")

In [ ]:
!python -m bitsandbytes

In [ ]:
import json
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from pathlib import Path

from transformers import AutoTokenizer

In [ ]:
# Устанавливаем scipy
!pip install scipy

# Теперь запускаем компиляцию заново
!git clone https://github.com/TimDettmers/bitsandbytes.git
%cd bitsandbytes
!CUDA_VERSION=128 make cuda12x
!python setup.py install
%cd ..

# Проверяем
import bitsandbytes
print("✅ bitsandbytes готов!")

In [ ]:
# ========== КОНФИГУРАЦИЯ ==========
CLASSIFIED_FILE  = '/kaggle/input/datasets/mayasirotkina/dataset/classified_corrections.json'
SOURCE_FILE      = '/kaggle/input/datasets/mayasirotkina/dataset/merged_results.json'
OUTPUT_DIR       = '/kaggle/working/vectors_output_v2'
MODEL_NAME       = 'Qwen/Qwen3-8B'

# Слои для анализа.
# Согласно рекомендации Neel Nanda — поздние средние слои наиболее информативны.
# Для Qwen3-8B (36 слоёв) это примерно слои 20-28.
# Мы прогоним несколько слоёв и сравним.
LAYERS_TO_TRY    = [20, 24, 28, -1]   # -1 = последний (35)
PRIMARY_LAYER    = 24                  # основной слой для финальных графиков

MAX_TOKENS       = 256
CONFIDENCE_FILTER = None   # None = все, 'high' = только уверенные
RANDOM_SEED      = 42
# ==================================

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
Path(OUTPUT_DIR).mkdir(exist_ok=True)

In [ ]:
# ============================================================
# 1. ЗАГРУЗКА ДАННЫХ (без изменений)
# ============================================================

def load_classified(classified_file, source_file):
    with open(classified_file, 'r', encoding='utf-8') as f:
        classified = json.load(f)
    with open(source_file, 'r', encoding='utf-8') as f:
        source_data = json.load(f)

    source_lookup = {}
    for idx, q in enumerate(source_data.get('questions', [])):
        q_id = q.get('id', idx + 1)
        source_lookup[q_id] = {
            'question': q.get('question', ''),
            'full_thinking': q.get('full_thinking', ''),
        }

    genuine_cases, stylistic_cases = [], []

    for q in classified.get('questions', []):
        q_id     = q['id']
        src      = source_lookup.get(q_id, {})
        thinking = src.get('full_thinking', '')
        question = src.get('question', q.get('question_preview', ''))

        if not thinking:
            continue

        for marker_data in q.get('markers', []):
            marker = marker_data['marker']
            for cls_entry in marker_data.get('classifications', []):
                cls        = cls_entry.get('classification', '')
                confidence = cls_entry.get('confidence', '')

                if CONFIDENCE_FILTER and confidence != CONFIDENCE_FILTER:
                    continue

                case = {
                    'question_id':        q_id,
                    'question_text':      question,
                    'thinking_text':      thinking,
                    'marker':             marker,
                    'occurrence_index':   cls_entry.get('occurrence_index', 1),
                    'position_in_tokens': cls_entry.get('position_in_tokens', 0),
                    'classification':     cls,
                    'confidence':         confidence,
                }

                if cls == 'genuine_correction':
                    genuine_cases.append(case)
                elif cls == 'stylistic_filler':
                    stylistic_cases.append(case)

    print(f"Genuine corrections : {len(genuine_cases)}")
    print(f"Stylistic fillers   : {len(stylistic_cases)}")
    return genuine_cases, stylistic_cases

In [ ]:
# ============================================================
# 2. ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ: токенизация и поиск маркера
# ============================================================

def find_marker_in_thinking(thinking_text, marker, occurrence_index):
    """
    Находит позицию нужного вхождения маркера в тексте.
    Возвращает (text_before, text_with_marker) или (None, None).

    Например, для thinking="let me think. wait, no. actually"
    и marker="actually", occurrence_index=1:
      text_before = "let me think . wait , no ."
      text_with_marker = "let me think . wait , no . actually"
    """
    tokens_list = re.findall(r'\w+|[^\w\s]', thinking_text.lower())
    occ_idx = occurrence_index - 1  # переводим в 0-based
    found_occ = 0
    actual_pos = None

    for i, tok in enumerate(tokens_list):
        if tok == marker.lower():
            if found_occ == occ_idx:
                actual_pos = i
                break
            found_occ += 1

    if actual_pos is None:
        return None, None, None

    text_before      = ' '.join(tokens_list[:actual_pos])
    text_with_marker = ' '.join(tokens_list[:actual_pos + 1])
    return text_before, text_with_marker, actual_pos


def build_prefix(tokenizer, question_text):
    """
    Строит chat-prefix с <think> вручную.
    Результат: '<|im_start|>user\n...question...<|im_end|>\n<|im_start|>assistant\n<think>\n'
    """
    messages = [{"role": "user", "content": question_text}]
    prefix = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    return prefix + "<think>\n"


def tokenize_prompt(tokenizer, prompt, max_tokens):
    """Токенизирует строку и обрезает до max_tokens с конца."""
    return tokenizer.encode(prompt)[-max_tokens:]

In [ ]:
def get_activation_at_position(model, token_ids, layer, position):
    """
    Прогоняет модель на token_ids и возвращает вектор активаций
    на указанной позиции токена в указанном слое.

    Как это работает:
    - Трансформер обрабатывает все токены параллельно.
    - На каждом слое каждый токен имеет свой вектор в residual stream.
    - Мы "подключаем хук" (hook) к нужному слою — это функция-перехватчик,
      которая срабатывает когда модель проходит через этот слой
      и сохраняет тензор активаций.
    - Потом берём активацию на нужной позиции токена.

    Аргументы:
      model      — загруженная HookedTransformer модель
      token_ids  — список токенов (чисел)
      layer      — номер слоя (0..n_layers-1, или отрицательный)
      position   — позиция токена в последовательности (-1 = последний)

    Возвращает:
      np.ndarray размером [d_model] — вектор активаций
    """
    n_layers = model.cfg.n_layers
    actual_layer = layer if layer >= 0 else n_layers + layer

    input_tensor = torch.tensor([token_ids], dtype=torch.long)
    captured = {}

    def hook_fn(value, hook):
        # value имеет форму [batch=1, seq_len, d_model]
        # .detach() — отключаем от графа вычислений (нам не нужны градиенты)
        # .cpu() — переносим на CPU для numpy
        # .float() — приводим к float32 (на случай если модель в float16)
        captured['resid'] = value.detach().cpu().float().numpy()

    hook_name = f'blocks.{actual_layer}.hook_resid_post'

    with torch.no_grad():  # отключаем вычисление градиентов — экономим память
        model.run_with_hooks(
            input_tensor,
            fwd_hooks=[(hook_name, hook_fn)]
        )

    # captured['resid'] имеет форму [1, seq_len, d_model]
    # берём batch=0, нужную позицию, все d_model измерения
    return captured['resid'][0, position, :]

In [ ]:
# ============================================================
# МЕТОД А: Comparative Activation Slicing
# ============================================================
#
# Идея:
#   Мы берём ОДИН прогон — полный текст с маркером.
#   Записываем активацию на двух позициях:
#     - позиция ПЕРЕД маркером ("Error State" — модель ещё не сказала wait)
#     - позиция САМОГО МАРКЕРА ("Correction State" — модель уже сказала wait)
#   Вектор = Acorr - Aerr
#
#   Затем усредняем по всем genuine случаям — это убирает шум
#   (конкретные значения слов, темы вопросов и т.д.)
#   и оставляет только то, что ОБЩЕГО у всех моментов коррекции.
#
#   Потом сравниваем: отличается ли этот вектор для genuine vs stylistic?

def extract_vector_method_a(model, tokenizer, case, layer, max_tokens):
    """
    Метод А: активация на маркере минус активация перед маркером.
    Один прогон на полный текст с маркером.

    Возвращает:
      (marker_activation, before_activation, delta_vector)
      или (None, None, None) если маркер не найден
    """
    thinking  = case['thinking_text']
    marker    = case['marker']
    occ_index = case['occurrence_index']

    text_before, text_with_marker, actual_pos = find_marker_in_thinking(
        thinking, marker, occ_index
    )
    if text_with_marker is None:
        return None, None, None

    # Строим один промпт: question + thinking включая маркер
    prefix = build_prefix(tokenizer, case['question_text'])
    full_prompt = prefix + text_with_marker
    token_ids = tokenize_prompt(tokenizer, full_prompt, max_tokens)

    if len(token_ids) < 5:
        return None, None, None

    # Активация на позиции последнего токена = маркер ("wait", "actually" и т.д.)
    # position=-1 это последний токен в последовательности
    act_marker = get_activation_at_position(model, token_ids, layer, position=-1)

    # Активация на позиции предпоследнего токена = токен перед маркером
    # position=-2 это второй с конца
    act_before = get_activation_at_position(model, token_ids, layer, position=-2)

    # Дельта = что изменилось в residual stream при переходе к маркеру коррекции
    delta = act_marker - act_before

    return act_marker, act_before, delta


def extract_all_method_a(model, tokenizer, cases, label, layer, max_tokens):
    """
    Прогоняет Метод А для всего списка случаев.
    Возвращает три массива: marker_vecs, before_vecs, delta_vecs
    """
    marker_vecs, before_vecs, delta_vecs = [], [], []
    skipped = 0

    print(f"\n[Метод А] Извлекаю [{label}] — {len(cases)} случаев, слой {layer}...")

    for i, case in enumerate(cases):
        try:
            act_m, act_b, delta = extract_vector_method_a(
                model, tokenizer, case, layer, max_tokens
            )
            if act_m is None:
                skipped += 1
                continue

            marker_vecs.append(act_m)
            before_vecs.append(act_b)
            delta_vecs.append(delta)

            if (i + 1) % 20 == 0 or i == 0:
                print(f"  [{i+1}/{len(cases)}] OK | delta norm={np.linalg.norm(delta):.4f}")

        except Exception as e:
            print(f"  [{i+1}/{len(cases)}] ОШИБКА: {e}")
            skipped += 1

    print(f"  Извлечено: {len(marker_vecs)}, пропущено: {skipped}")

    if not marker_vecs:
        return np.array([]), np.array([]), np.array([])

    return np.vstack(marker_vecs), np.vstack(before_vecs), np.vstack(delta_vecs)

In [ ]:
# ============================================================
# МЕТОД Б: Counterfactual Activation Addition (CAA)
# ============================================================
#
# Идея:
#   Только для GENUINE случаев.
#   У нас есть настоящая коррекция: модель сказала "wait" и исправилась.
#   Мы хотим узнать: что было бы если бы модель НЕ исправилась?
#
#   correction_run: [текст ... wait]  ← реальный ход событий
#   counterfactual: [текст ... ???]   ← мы подставляем другой токен вместо wait
#
#   Как выбрать "???":
#   Смотрим на логиты (вероятности) модели на позиции перед маркером.
#   Берём токен с наибольшей вероятностью, который НЕ является маркером.
#   Это "продолжение ошибки" — что модель сказала бы если бы не исправлялась.
#
#   Вектор = активация(correction_run на маркере)
#            - активация(counterfactual на том же месте)
#
#   Это более "чистый" сигнал коррекции чем Метод А,
#   потому что мы контролируем контекст — он одинаковый, меняется только токен.

def get_top_non_marker_token(model, tokenizer, token_ids_before_marker, marker):
    """
    Смотрит какой токен модель предсказала бы вместо маркера.

    Как это работает:
    - Прогоняем модель на тексте БЕЗ маркера (до него)
    - Смотрим на логиты (числа для каждого токена в словаре)
    - Берём топ-5 наиболее вероятных токенов
    - Возвращаем первый который не является нашим маркером
    """
    marker_token_ids = set(tokenizer.encode(marker, add_special_tokens=False))
    # Для маркеров с пробелом в начале
    marker_token_ids |= set(tokenizer.encode(' ' + marker, add_special_tokens=False))

    input_tensor = torch.tensor([token_ids_before_marker], dtype=torch.long)

    with torch.no_grad():
        logits = model(input_tensor)  # форма: [1, seq_len, vocab_size]

    # Логиты последнего токена — это предсказание СЛЕДУЮЩЕГО токена
    last_logits = logits[0, -1, :]  # форма: [vocab_size]

    # Берём топ-10 токенов по вероятности
    top_tokens = torch.topk(last_logits, k=10).indices.tolist()

    # Возвращаем первый, который не является маркером коррекции
    for tok_id in top_tokens:
        if tok_id not in marker_token_ids:
            return tok_id

    # Запасной вариант — просто второй по вероятности токен
    return top_tokens[1]


def extract_vector_method_b(model, tokenizer, case, layer, max_tokens):
    """
    Метод Б: correction_run - counterfactual_run на позиции маркера.

    Возвращает вектор или None.
    """
    thinking  = case['thinking_text']
    marker    = case['marker']
    occ_index = case['occurrence_index']

    text_before, text_with_marker, _ = find_marker_in_thinking(
        thinking, marker, occ_index
    )
    if text_with_marker is None:
        return None

    prefix = build_prefix(tokenizer, case['question_text'])

    # --- Прогон 1: correction_run (с маркером) ---
    correction_prompt = prefix + text_with_marker
    correction_tokens = tokenize_prompt(tokenizer, correction_prompt, max_tokens)

    if len(correction_tokens) < 5:
        return None

    # Активация на последнем токене (= маркер "wait"/"actually")
    act_correction = get_activation_at_position(
        model, correction_tokens, layer, position=-1
    )

    # --- Прогон 2: находим counterfactual токен ---
    control_prompt = prefix + text_before
    control_tokens = tokenize_prompt(tokenizer, control_prompt, max_tokens)

    if len(control_tokens) < 5:
        return None

    # Какой токен модель предсказала бы вместо маркера?
    counterfactual_token_id = get_top_non_marker_token(
        model, tokenizer, control_tokens, marker
    )

    # Создаём counterfactual последовательность:
    # берём control_tokens и добавляем counterfactual токен
    counterfactual_tokens = list(control_tokens) + [counterfactual_token_id]
    # Обрезаем если вышли за max_tokens
    counterfactual_tokens = counterfactual_tokens[-max_tokens:]

    # Активация на последнем токене counterfactual (= "продолжение ошибки")
    act_counterfactual = get_activation_at_position(
        model, counterfactual_tokens, layer, position=-1
    )

    # Вектор = что отличает момент коррекции от продолжения ошибки
    vector = act_correction - act_counterfactual
    return vector


def extract_all_method_b(model, tokenizer, cases, label, layer, max_tokens):
    """
    Прогоняет Метод Б для всего списка случаев.
    Работает только для genuine_cases (для stylistic тоже можно, для сравнения).
    """
    vectors = []
    skipped = 0

    print(f"\n[Метод Б] Извлекаю [{label}] — {len(cases)} случаев, слой {layer}...")

    for i, case in enumerate(cases):
        try:
            vec = extract_vector_method_b(model, tokenizer, case, layer, max_tokens)
            if vec is None:
                skipped += 1
                continue

            vectors.append(vec)

            if (i + 1) % 20 == 0 or i == 0:
                print(f"  [{i+1}/{len(cases)}] OK | norm={np.linalg.norm(vec):.4f}")

        except Exception as e:
            print(f"  [{i+1}/{len(cases)}] ОШИБКА: {e}")
            skipped += 1

    print(f"  Извлечено: {len(vectors)}, пропущено: {skipped}")

    if not vectors:
        return np.array([])

    return np.vstack(vectors)

In [ ]:
# ============================================================
# 4. ВИЗУАЛИЗАЦИЯ И АНАЛИЗ
# ============================================================

def plot_pca_comparison(genuine_vecs, stylistic_vecs, title, save_path):
    """
    PCA: проецируем высокомерные векторы на 2D плоскость.
    Если genuine и stylistic кластеризуются отдельно — есть структура.
    """
    if len(genuine_vecs) == 0 or len(stylistic_vecs) == 0:
        print(f"  Пропускаю {title}: нет данных")
        return

    all_vecs = np.vstack([genuine_vecs, stylistic_vecs])
    labels   = ['genuine'] * len(genuine_vecs) + ['stylistic'] * len(stylistic_vecs)

    scaler = StandardScaler()
    all_scaled = scaler.fit_transform(all_vecs)

    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    coords = pca.fit_transform(all_scaled)

    var1, var2 = pca.explained_variance_ratio_[:2] * 100

    fig, ax = plt.subplots(figsize=(9, 7))
    colors  = {'genuine': '#e74c3c', 'stylistic': '#3498db'}
    markers = {'genuine': 'o',       'stylistic': 's'}

    for lbl in ['genuine', 'stylistic']:
        idxs = [i for i, l in enumerate(labels) if l == lbl]
        ax.scatter(coords[idxs, 0], coords[idxs, 1],
                   c=colors[lbl], marker=markers[lbl],
                   alpha=0.7, s=60, label=f'{lbl} (n={len(idxs)})')

    ax.set_xlabel(f'PC1 ({var1:.1f}%)')
    ax.set_ylabel(f'PC2 ({var2:.1f}%)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"  Сохранено: {save_path}")


def plot_layer_analysis(genuine_vecs_by_layer, stylistic_vecs_by_layer, layers, save_path):
    """
    График: косинусное сходство между центроидами genuine и stylistic
    для каждого слоя.

    Чем МЕНЬШЕ сходство — тем лучше разделение на этом слое.
    Это помогает найти "информативный" слой.
    """
    sims = []
    for layer in layers:
        g = genuine_vecs_by_layer.get(layer)
        s = stylistic_vecs_by_layer.get(layer)
        if g is None or s is None or len(g) == 0 or len(s) == 0:
            sims.append(None)
            continue
        mean_g = g.mean(axis=0, keepdims=True)
        mean_s = s.mean(axis=0, keepdims=True)
        sim = cosine_similarity(mean_g, mean_s)[0, 0]
        sims.append(sim)

    valid = [(l, s) for l, s in zip(layers, sims) if s is not None]
    if not valid:
        return

    ls, ss = zip(*valid)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(ls, ss, 'o-', color='#2ecc71', linewidth=2, markersize=8)
    ax.axhline(0, color='gray', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Слой')
    ax.set_ylabel('Косинусное сходство центроидов')
    ax.set_title('Разделение genuine vs stylistic по слоям\n(меньше = лучше)')
    ax.grid(True, alpha=0.3)

    # Отмечаем лучший слой
    best_idx = np.argmin(ss)
    ax.annotate(f'Лучший: слой {ls[best_idx]}\n({ss[best_idx]:.3f})',
                xy=(ls[best_idx], ss[best_idx]),
                xytext=(ls[best_idx]+1, ss[best_idx]+0.05),
                fontsize=10, color='red')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"  Сохранено: {save_path}")


def run_linear_probe(genuine_vecs, stylistic_vecs, label=''):
    """
    Линейный классификатор (logistic regression) поверх векторов.

    Если PCA не даёт визуального разделения, это не значит что его нет —
    оно может быть в направлениях которые PCA не захватывает.
    Logistic regression ищет любую линейную границу в полном пространстве.

    Accuracy > 0.6 — есть структура.
    Accuracy ~0.5 — случайный угадыватель, структуры нет.
    """
    if len(genuine_vecs) < 10 or len(stylistic_vecs) < 10:
        print(f"  Слишком мало данных для линейного пробинга")
        return

    X = np.vstack([genuine_vecs, stylistic_vecs])
    y = np.array([1] * len(genuine_vecs) + [0] * len(stylistic_vecs))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    # 5-fold cross-validation: честная оценка без переобучения
    scores = cross_val_score(clf, X_scaled, y, cv=5, scoring='accuracy')

    print(f"\n  Линейный пробинг [{label}]:")
    print(f"  Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")
    print(f"  (baseline случайного угадывания: {max(len(genuine_vecs), len(stylistic_vecs)) / len(y):.3f})")

    return scores.mean()

In [ ]:
# ============================================================
# 5. MAIN
# ============================================================

def main():
    print("=" * 65)
    print("ИЗВЛЕЧЕНИЕ SELF-CORRECTION ВЕКТОРОВ v2")
    print("=" * 65)

    # --- Загрузка данных ---
    genuine_cases, stylistic_cases = load_classified(CLASSIFIED_FILE, SOURCE_FILE)

    # --- Загрузка модели ---
    print(f"\nЗагружаю модель {MODEL_NAME}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Устройство: {device}")

    model = HookedTransformer.from_pretrained(
        MODEL_NAME,
        device=device,
        dtype=torch.float16,
        trust_remote_code=True,
    )
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    print(f"Модель: {model.cfg.n_layers} слоёв, d_model={model.cfg.d_model}")

    # Проверка chat template
    test_prefix = build_prefix(tokenizer, "test")
    print("Префикс заканчивается на:", repr(test_prefix[-20:]))
    # Должно быть: '...assistant\n<think>\n'

    # ========================================================
    # МЕТОД А: marker vs before_marker
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД А: Comparative Activation Slicing")
    print("=" * 50)

    # Анализ по нескольким слоям
    genuine_marker_by_layer   = {}
    stylistic_marker_by_layer = {}

    for layer in LAYERS_TO_TRY:
        print(f"\n--- Слой {layer} ---")

        g_marker, g_before, g_delta = extract_all_method_a(
            model, tokenizer, genuine_cases, 'genuine', layer, MAX_TOKENS
        )
        s_marker, s_before, s_delta = extract_all_method_a(
            model, tokenizer, stylistic_cases, 'stylistic', layer, MAX_TOKENS
        )

        if len(g_marker) > 0 and len(s_marker) > 0:
            genuine_marker_by_layer[layer]   = g_marker
            stylistic_marker_by_layer[layer] = s_marker

            # Быстрая проверка разделимости
            mean_g = g_marker.mean(axis=0, keepdims=True)
            mean_s = s_marker.mean(axis=0, keepdims=True)
            sim = cosine_similarity(mean_g, mean_s)[0, 0]
            print(f"  Косинусное сходство центроидов: {sim:.4f}")

    # График по слоям
    plot_layer_analysis(
        genuine_marker_by_layer,
        stylistic_marker_by_layer,
        LAYERS_TO_TRY,
        f'{OUTPUT_DIR}/method_a_layer_analysis.png'
    )

    # PCA для основного слоя
    g_main = genuine_marker_by_layer.get(PRIMARY_LAYER)
    s_main = stylistic_marker_by_layer.get(PRIMARY_LAYER)

    if g_main is not None and s_main is not None:
        plot_pca_comparison(
            g_main, s_main,
            f'Метод А: активация на маркере (слой {PRIMARY_LAYER})',
            f'{OUTPUT_DIR}/method_a_pca_layer{PRIMARY_LAYER}.png'
        )
        run_linear_probe(g_main, s_main, f'Метод А, слой {PRIMARY_LAYER}')

    # ========================================================
    # МЕТОД Б: correction vs counterfactual
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД Б: Counterfactual Activation Addition")
    print("=" * 50)

    # Метод Б для genuine
    genuine_caa = extract_all_method_b(
        model, tokenizer, genuine_cases, 'genuine', PRIMARY_LAYER, MAX_TOKENS
    )
    # Метод Б для stylistic (для сравнения)
    stylistic_caa = extract_all_method_b(
        model, tokenizer, stylistic_cases, 'stylistic', PRIMARY_LAYER, MAX_TOKENS
    )

    if len(genuine_caa) > 0 and len(stylistic_caa) > 0:
        plot_pca_comparison(
            genuine_caa, stylistic_caa,
            f'Метод Б: correction - counterfactual (слой {PRIMARY_LAYER})',
            f'{OUTPUT_DIR}/method_b_pca_layer{PRIMARY_LAYER}.png'
        )
        run_linear_probe(genuine_caa, stylistic_caa, f'Метод Б, слой {PRIMARY_LAYER}')

        # Сохраняем
        np.savez(
            f'{OUTPUT_DIR}/vectors_method_b.npz',
            genuine_vectors=genuine_caa,
            stylistic_vectors=stylistic_caa
        )
        print(f"  Векторы Метода Б сохранены")

    print(f"\n{'=' * 65}")
    print(f"ГОТОВО. Результаты в: {OUTPUT_DIR}")
    print(f"{'=' * 65}")


if __name__ == "__main__":
    main()

In [ ]:
def main_with_probing():
    print("=" * 65)
    print("ИЗВЛЕЧЕНИЕ SELF-CORRECTION ВЕКТОРОВ И ЛИНЕЙНЫЙ ПРОБИНГ")
    print("=" * 65)

    # --- Загрузка данных ---
    genuine_cases, stylistic_cases = load_classified(CLASSIFIED_FILE, SOURCE_FILE)

    # --- Загрузка модели Qwen3-8B ---
    print(f"\nЗагружаю Qwen3-8B...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Устройство: {device}")

    # Загружаем через transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from transformer_lens.HookedTransformerConfig import HookedTransformerConfig
    from transformer_lens import HookedTransformer

    print("Шаг 1: Загрузка через transformers...")
    hf_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-8B",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="auto",
        low_cpu_mem_usage=True
    )

    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-8B",
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Шаг 2: Создание конфига для Qwen3-8B...")
    # Создаем конфиг вручную
    cfg = HookedTransformerConfig(
        n_layers=36,
        d_model=4096,
        d_head=128,
        n_heads=32,
        d_mlp=14336,
        d_vocab=152064,  # размер словаря Qwen3
        n_ctx=32768,     # максимальная длина контекста
        act_fn="silu",
        normalization_type="RMS",
        positional_embedding_type="rotary",
        rotary_dim=128,
        rotary_base=1000000,
        attn_only=False,
        model_name="Qwen/Qwen3-8B",
        device=device,
        dtype=torch.float16,
        seed=42,
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_mlp_in=True
    )

    print("Шаг 3: Создание HookedTransformer...")
    # Создаем модель с конфигом
    model = HookedTransformer(cfg, tokenizer)

    print("Шаг 4: Загрузка весов...")
    # Загружаем веса из hf_model
    model.load_and_process_state_dict(
        hf_model.state_dict(),
        fold_ln=False,
        center_writing_weights=False,
        center_unembed=False
    )

    model = model.to(device)
    model.eval()

    print(f"Модель загружена!")
    print(f"  Слоёв: {model.cfg.n_layers}")
    print(f"  d_model: {model.cfg.d_model}")
    print(f"  Тип: {model.cfg.model_name}")

    # ========================================================
    # МЕТОД А: marker vs before_marker - С РАСШИРЕННЫМ ПРОБИНГОМ
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД А: Comparative Activation Slicing")
    print("=" * 50)

    # Словари для хранения всех типов векторов
    genuine_vectors_by_layer = {}
    stylistic_vectors_by_layer = {}

    for layer in LAYERS_TO_TRY:
        print(f"\n--- Слой {layer} ---")

        g_marker, g_before, g_delta = extract_all_method_a(
            model, tokenizer, genuine_cases, 'genuine', layer, MAX_TOKENS
        )
        s_marker, s_before, s_delta = extract_all_method_a(
            model, tokenizer, stylistic_cases, 'stylistic', layer, MAX_TOKENS
        )

        if len(g_marker) > 0 and len(s_marker) > 0:
            genuine_vectors_by_layer[layer] = {
                'marker': g_marker,
                'before': g_before,
                'delta': g_delta
            }
            stylistic_vectors_by_layer[layer] = {
                'marker': s_marker,
                'before': s_before,
                'delta': s_delta
            }

    # Анализ для основного слоя с сравнением источников
    if PRIMARY_LAYER in genuine_vectors_by_layer:
        compare_activation_sources(
            genuine_vectors_by_layer[PRIMARY_LAYER],
            stylistic_vectors_by_layer[PRIMARY_LAYER],
            PRIMARY_LAYER
        )

    # ========================================================
    # МЕТОД Б: correction vs counterfactual
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД Б: Counterfactual Activation Addition")
    print("=" * 50)

    # Метод Б для genuine
    genuine_caa = extract_all_method_b(
        model, tokenizer, genuine_cases, 'genuine', PRIMARY_LAYER, MAX_TOKENS
    )
    # Метод Б для stylistic (для сравнения)
    stylistic_caa = extract_all_method_b(
        model, tokenizer, stylistic_cases, 'stylistic', PRIMARY_LAYER, MAX_TOKENS
    )

    if len(genuine_caa) > 0 and len(stylistic_caa) > 0:
        # Расширенный пробинг для метода Б
        run_linear_probe_detailed(
            genuine_caa, stylistic_caa,
            label=f'Метод Б, слой {PRIMARY_LAYER}',
            test_size=0.2
        )

        # Сравнение с методом А (дельта)
        if PRIMARY_LAYER in genuine_vectors_by_layer:
            g_delta = genuine_vectors_by_layer[PRIMARY_LAYER]['delta']
            s_delta = stylistic_vectors_by_layer[PRIMARY_LAYER]['delta']

            print("\n" + "=" * 50)
            print("СРАВНЕНИЕ МЕТОДОВ А (дельта) и Б (CAA)")
            print("=" * 50)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # ROC кривые для обоих методов
            from sklearn.metrics import roc_curve

            # Метод А
            X_a = np.vstack([g_delta, s_delta])
            y_a = np.array([1] * len(g_delta) + [0] * len(s_delta))
            X_a_scaled = StandardScaler().fit_transform(X_a)
            X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
                X_a_scaled, y_a, test_size=0.2, random_state=RANDOM_SEED, stratify=y_a
            )

            lr_a = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
            lr_a.fit(X_a_train, y_a_train)
            y_a_proba = lr_a.predict_proba(X_a_test)[:, 1]
            fpr_a, tpr_a, _ = roc_curve(y_a_test, y_a_proba)
            auc_a = roc_auc_score(y_a_test, y_a_proba)

            # Метод Б
            X_b = np.vstack([genuine_caa, stylistic_caa])
            y_b = np.array([1] * len(genuine_caa) + [0] * len(stylistic_caa))
            X_b_scaled = StandardScaler().fit_transform(X_b)
            X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
                X_b_scaled, y_b, test_size=0.2, random_state=RANDOM_SEED, stratify=y_b
            )

            lr_b = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
            lr_b.fit(X_b_train, y_b_train)
            y_b_proba = lr_b.predict_proba(X_b_test)[:, 1]
            fpr_b, tpr_b, _ = roc_curve(y_b_test, y_b_proba)
            auc_b = roc_auc_score(y_b_test, y_b_proba)

            # График 1: ROC кривые
            axes[0].plot(fpr_a, tpr_a, label=f'Метод А (дельта) AUC={auc_a:.3f}', linewidth=2)
            axes[0].plot(fpr_b, tpr_b, label=f'Метод Б (CAA) AUC={auc_b:.3f}', linewidth=2)
            axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
            axes[0].set_xlabel('False Positive Rate')
            axes[0].set_ylabel('True Positive Rate')
            axes[0].set_title('ROC Curves Comparison')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)

            # График 2: Сравнение accuracy
            methods = ['Метод А (дельта)', 'Метод Б (CAA)']
            accuracies = [lr_a.score(X_a_test, y_a_test), lr_b.score(X_b_test, y_b_test)]

            axes[1].bar(methods, accuracies, color=['#3498db', '#e74c3c'], alpha=0.7)
            axes[1].set_ylabel('Test Accuracy')
            axes[1].set_title('Accuracy Comparison')
            axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

            for i, v in enumerate(accuracies):
                axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

            axes[1].set_ylim([0, 1])
            axes[1].grid(True, alpha=0.3)

            plt.suptitle(f'Сравнение методов извлечения векторов (слой {PRIMARY_LAYER})')
            plt.tight_layout()
            plt.savefig(f'{OUTPUT_DIR}/method_comparison_layer{PRIMARY_LAYER}.png', dpi=150)
            plt.show()

    print(f"\n{'='*65}")
    print(f"ГОТОВО. Результаты в: {OUTPUT_DIR}")
    print(f"{'='*65}")


if __name__ == "__main__":
    # main()  # старая версия
    main_with_probing()  # новая версия с расширенным пробингом

In [ ]:
def main_with_probing():
    print("=" * 65)
    print("ИЗВЛЕЧЕНИЕ SELF-CORRECTION ВЕКТОРОВ И ЛИНЕЙНЫЙ ПРОБИНГ")
    print("=" * 65)

    # --- Загрузка данных ---
    genuine_cases, stylistic_cases = load_classified(CLASSIFIED_FILE, SOURCE_FILE)

    # --- Загрузка модели Qwen3-8B с оптимизацией памяти ---
    print(f"\nЗагружаю Qwen3-8B с батчевой обработкой...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Устройство: {device}")

    from transformers import AutoModelForCausalLM, AutoTokenizer
    from transformer_lens.HookedTransformerConfig import HookedTransformerConfig
    from transformer_lens import HookedTransformer
    import gc
    import psutil

    def print_memory_usage():
        """Вспомогательная функция для отслеживания памяти"""
        if torch.cuda.is_available():
            print(f"  GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, "
                  f"{torch.cuda.memory_reserved()/1e9:.2f} GB reserved")
        print(f"  RAM: {psutil.Process().memory_info().rss / 1e9:.2f} GB")

    print("Шаг 1: Загрузка через transformers с CPU offloading...")

    # Очищаем память перед загрузкой
    gc.collect()
    torch.cuda.empty_cache()

    # Загружаем модель с CPU offloading для экономии GPU памяти
    hf_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen3-8B",
        torch_dtype=torch.float16,
        trust_remote_code=True,
        device_map="cpu",  # Сначала на CPU
        low_cpu_mem_usage=True
    )

    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-8B",
        trust_remote_code=True
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Шаг 2: Создание конфига для Qwen3-8B...")
    # Создаем конфиг вручную
    cfg = HookedTransformerConfig(
        n_layers=36,
        d_model=4096,
        d_head=128,
        n_heads=32,
        d_mlp=14336,
        d_vocab=152064,
        n_ctx=32768,
        act_fn="silu",
        normalization_type="RMS",  # Важно: "RMS" а не "RMSNorm"
        positional_embedding_type="rotary",
        rotary_dim=128,
        rotary_base=1000000,
        attn_only=False,
        model_name="Qwen/Qwen3-8B",
        device=device,
        dtype=torch.float16,
        seed=42,
        use_attn_result=True,
        use_split_qkv_input=True,
        use_hook_mlp_in=True
    )

    print("Шаг 3: Создание HookedTransformer на CPU...")
    # Создаем модель на CPU сначала
    model = HookedTransformer(cfg, tokenizer)
    model = model.to('cpu')

    print("Шаг 4: Загрузка весов на CPU...")
    # Загружаем веса на CPU
    state_dict = hf_model.state_dict()

    model.load_and_process_state_dict(
        state_dict,
        fold_ln=False,
        center_writing_weights=False,
        center_unembed=False
    )

    # Освобождаем transformers модель
    del hf_model
    gc.collect()

    print("Шаг 5: Перемещаем модель на GPU частями...")
    # Перемещаем на GPU по частям
    model = model.to(device)
    gc.collect()
    torch.cuda.empty_cache()

    model.eval()

    print(f"Модель загружена!")
    print(f"  Слоёв: {model.cfg.n_layers}")
    print(f"  d_model: {model.cfg.d_model}")
    print_memory_usage()

    # ========================================================
    # МЕТОД А: marker vs before_marker - С РАСШИРЕННЫМ ПРОБИНГОМ И БАТЧЕВОЙ ОБРАБОТКОЙ
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД А: Comparative Activation Slicing")
    print("=" * 50)

    # Параметры батчевой обработки
    BATCH_SIZE = 5  # Маленькие батчи для экономии памяти

    # Словари для хранения всех типов векторов
    genuine_vectors_by_layer = {}
    stylistic_vectors_by_layer = {}

    for layer in LAYERS_TO_TRY:
        print(f"\n--- Слой {layer} ---")

        # Списки для сбора результатов
        g_marker_list, g_before_list, g_delta_list = [], [], []
        s_marker_list, s_before_list, s_delta_list = [], [], []

        # Обрабатываем genuine случаи батчами
        total_batches = (len(genuine_cases) + BATCH_SIZE - 1) // BATCH_SIZE
        for batch_idx in range(total_batches):
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min(start_idx + BATCH_SIZE, len(genuine_cases))
            batch = genuine_cases[start_idx:end_idx]

            print(f"  Батч genuine {batch_idx + 1}/{total_batches} ({len(batch)} примеров)")

            for i, case in enumerate(batch):
                try:
                    act_m, act_b, delta = extract_vector_method_a(
                        model, tokenizer, case, layer, MAX_TOKENS
                    )
                    if act_m is not None:
                        g_marker_list.append(act_m)
                        g_before_list.append(act_b)
                        g_delta_list.append(delta)

                    # Прогресс внутри батча
                    if (i + 1) % 2 == 0:
                        print(f"    Обработано {i + 1}/{len(batch)} в батче")

                except Exception as e:
                    print(f"    Ошибка в примере {i}: {e}")
                    continue

            # Очищаем память после каждого батча
            gc.collect()
            torch.cuda.empty_cache()
            print(f"    Память после батча: {torch.cuda.memory_allocated()/1e9:.2f} GB")

        # Обрабатываем stylistic случаи батчами
        total_batches = (len(stylistic_cases) + BATCH_SIZE - 1) // BATCH_SIZE
        for batch_idx in range(total_batches):
            start_idx = batch_idx * BATCH_SIZE
            end_idx = min(start_idx + BATCH_SIZE, len(stylistic_cases))
            batch = stylistic_cases[start_idx:end_idx]

            print(f"  Батч stylistic {batch_idx + 1}/{total_batches} ({len(batch)} примеров)")

            for i, case in enumerate(batch):
                try:
                    act_m, act_b, delta = extract_vector_method_a(
                        model, tokenizer, case, layer, MAX_TOKENS
                    )
                    if act_m is not None:
                        s_marker_list.append(act_m)
                        s_before_list.append(act_b)
                        s_delta_list.append(delta)

                    if (i + 1) % 2 == 0:
                        print(f"    Обработано {i + 1}/{len(batch)} в батче")

                except Exception as e:
                    print(f"    Ошибка в примере {i}: {e}")
                    continue

            gc.collect()
            torch.cuda.empty_cache()
            print(f"    Память после батча: {torch.cuda.memory_allocated()/1e9:.2f} GB")

        # Сохраняем результаты для этого слоя
        if len(g_marker_list) > 0 and len(s_marker_list) > 0:
            genuine_vectors_by_layer[layer] = {
                'marker': np.vstack(g_marker_list),
                'before': np.vstack(g_before_list),
                'delta': np.vstack(g_delta_list)
            }
            stylistic_vectors_by_layer[layer] = {
                'marker': np.vstack(s_marker_list),
                'before': np.vstack(s_before_list),
                'delta': np.vstack(s_delta_list)
            }

            print(f"\n  Результаты для слоя {layer}:")
            print(f"    Genuine извлечено: {len(g_marker_list)}")
            print(f"    Stylistic извлечено: {len(s_marker_list)}")

            # Быстрая проверка разделимости
            mean_g = np.mean(g_marker_list, axis=0)
            mean_s = np.mean(s_marker_list, axis=0)
            sim = np.dot(mean_g, mean_s) / (np.linalg.norm(mean_g) * np.linalg.norm(mean_s))
            print(f"    Косинусное сходство центроидов: {sim:.4f}")

    # Анализ для основного слоя с сравнением источников
    if PRIMARY_LAYER in genuine_vectors_by_layer:
        print(f"\nАнализ для основного слоя {PRIMARY_LAYER}...")
        compare_activation_sources(
            genuine_vectors_by_layer[PRIMARY_LAYER],
            stylistic_vectors_by_layer[PRIMARY_LAYER],
            PRIMARY_LAYER
        )

    # ========================================================
    # МЕТОД Б: correction vs counterfactual (тоже с батчами)
    # ========================================================
    print("\n" + "=" * 50)
    print("МЕТОД Б: Counterfactual Activation Addition")
    print("=" * 50)

    # Очищаем память перед методом Б
    gc.collect()
    torch.cuda.empty_cache()

    # Списки для сбора результатов метода Б
    genuine_caa_list = []
    stylistic_caa_list = []

    # Метод Б для genuine с батчами
    print(f"\nИзвлечение CAA векторов для genuine (слой {PRIMARY_LAYER})...")
    total_batches = (len(genuine_cases) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_idx in range(total_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(genuine_cases))
        batch = genuine_cases[start_idx:end_idx]

        print(f"  Батч {batch_idx + 1}/{total_batches} ({len(batch)} примеров)")

        for i, case in enumerate(batch):
            try:
                vec = extract_vector_method_b(
                    model, tokenizer, case, PRIMARY_LAYER, MAX_TOKENS
                )
                if vec is not None:
                    genuine_caa_list.append(vec)

                if (i + 1) % 2 == 0:
                    print(f"    Обработано {i + 1}/{len(batch)}")

            except Exception as e:
                print(f"    Ошибка: {e}")
                continue

        gc.collect()
        torch.cuda.empty_cache()

    # Метод Б для stylistic с батчами
    print(f"\nИзвлечение CAA векторов для stylistic (слой {PRIMARY_LAYER})...")
    total_batches = (len(stylistic_cases) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_idx in range(total_batches):
        start_idx = batch_idx * BATCH_SIZE
        end_idx = min(start_idx + BATCH_SIZE, len(stylistic_cases))
        batch = stylistic_cases[start_idx:end_idx]

        print(f"  Батч {batch_idx + 1}/{total_batches} ({len(batch)} примеров)")

        for i, case in enumerate(batch):
            try:
                vec = extract_vector_method_b(
                    model, tokenizer, case, PRIMARY_LAYER, MAX_TOKENS
                )
                if vec is not None:
                    stylistic_caa_list.append(vec)

                if (i + 1) % 2 == 0:
                    print(f"    Обработано {i + 1}/{len(batch)}")

            except Exception as e:
                print(f"    Ошибка: {e}")
                continue

        gc.collect()
        torch.cuda.empty_cache()

    # Конвертируем в numpy массивы
    if len(genuine_caa_list) > 0 and len(stylistic_caa_list) > 0:
        genuine_caa = np.vstack(genuine_caa_list)
        stylistic_caa = np.vstack(stylistic_caa_list)

        print(f"\nCAA векторы извлечены:")
        print(f"  Genuine: {len(genuine_caa)}")
        print(f"  Stylistic: {len(stylistic_caa)}")

        # Расширенный пробинг для метода Б
        run_linear_probe_detailed(
            genuine_caa, stylistic_caa,
            label=f'Метод Б, слой {PRIMARY_LAYER}',
            test_size=0.2
        )

        # Сравнение с методом А (дельта)
        if PRIMARY_LAYER in genuine_vectors_by_layer:
            g_delta = genuine_vectors_by_layer[PRIMARY_LAYER]['delta']
            s_delta = stylistic_vectors_by_layer[PRIMARY_LAYER]['delta']

            print("\n" + "=" * 50)
            print("СРАВНЕНИЕ МЕТОДОВ А (дельта) и Б (CAA)")
            print("=" * 50)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # ROC кривые для обоих методов
            from sklearn.metrics import roc_curve

            # Метод А
            X_a = np.vstack([g_delta, s_delta])
            y_a = np.array([1] * len(g_delta) + [0] * len(s_delta))
            X_a_scaled = StandardScaler().fit_transform(X_a)
            X_a_train, X_a_test, y_a_train, y_a_test = train_test_split(
                X_a_scaled, y_a, test_size=0.2, random_state=RANDOM_SEED, stratify=y_a
            )

            lr_a = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
            lr_a.fit(X_a_train, y_a_train)
            y_a_proba = lr_a.predict_proba(X_a_test)[:, 1]
            fpr_a, tpr_a, _ = roc_curve(y_a_test, y_a_proba)
            auc_a = roc_auc_score(y_a_test, y_a_proba)

            # Метод Б
            X_b = np.vstack([genuine_caa, stylistic_caa])
            y_b = np.array([1] * len(genuine_caa) + [0] * len(stylistic_caa))
            X_b_scaled = StandardScaler().fit_transform(X_b)
            X_b_train, X_b_test, y_b_train, y_b_test = train_test_split(
                X_b_scaled, y_b, test_size=0.2, random_state=RANDOM_SEED, stratify=y_b
            )

            lr_b = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
            lr_b.fit(X_b_train, y_b_train)
            y_b_proba = lr_b.predict_proba(X_b_test)[:, 1]
            fpr_b, tpr_b, _ = roc_curve(y_b_test, y_b_proba)
            auc_b = roc_auc_score(y_b_test, y_b_proba)

            # График 1: ROC кривые
            axes[0].plot(fpr_a, tpr_a, label=f'Метод А (дельта) AUC={auc_a:.3f}', linewidth=2)
            axes[0].plot(fpr_b, tpr_b, label=f'Метод Б (CAA) AUC={auc_b:.3f}', linewidth=2)
            axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
            axes[0].set_xlabel('False Positive Rate')
            axes[0].set_ylabel('True Positive Rate')
            axes[0].set_title('ROC Curves Comparison')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)

            # График 2: Сравнение accuracy
            methods = ['Метод А (дельта)', 'Метод Б (CAA)']
            accuracies = [lr_a.score(X_a_test, y_a_test), lr_b.score(X_b_test, y_b_test)]

            axes[1].bar(methods, accuracies, color=['#3498db', '#e74c3c'], alpha=0.7)
            axes[1].set_ylabel('Test Accuracy')
            axes[1].set_title('Accuracy Comparison')
            axes[1].axhline(y=0.5, color='red', linestyle='--', alpha=0.5)

            for i, v in enumerate(accuracies):
                axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', va='bottom')

            axes[1].set_ylim([0, 1])
            axes[1].grid(True, alpha=0.3)

            plt.suptitle(f'Сравнение методов извлечения векторов (слой {PRIMARY_LAYER})')
            plt.tight_layout()
            plt.savefig(f'{OUTPUT_DIR}/method_comparison_layer{PRIMARY_LAYER}.png', dpi=150)
            plt.show()

        # Сохраняем векторы
        np.savez(
            f'{OUTPUT_DIR}/vectors_method_b.npz',
            genuine_vectors=genuine_caa,
            stylistic_vectors=stylistic_caa
        )
        print(f"\n  Векторы Метода Б сохранены в {OUTPUT_DIR}/vectors_method_b.npz")

    print(f"\n{'='*65}")
    print(f"ГОТОВО. Результаты в: {OUTPUT_DIR}")
    print(f"{'='*65}")

In [ ]:
!pip install bitsandbytes

In [ ]:
# ШАГ 1: Полная зачистка и установка правильных версий
!pip uninstall -y transformers accelerate torch bitsandbytes numpy
!rm -rf ~/.cache/huggingface/hub/

# Устанавливаем transformers 4.51.0 (нужная для Qwen3)
!pip install transformers==4.51.0
!pip install accelerate==0.33.0
!pip install torch==2.4.0 --index-url https://download.pytorch.org/whl/cu121
!pip install numpy==1.24.3
!pip install scipy

# Перезапуск ядра
import os
os.kill(os.getpid(), 9)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

def run_linear_probe(genuine_vecs, stylistic_vecs, layer):
    """Линейный пробинг на активациях (не на разности!)"""
    print("\n" + "=" * 60)
    print(f"ЛИНЕЙНЫЙ ПРОБИНГ (слой {layer})")
    print("=" * 60)

    # Подготовка данных (используем marker_vecs, не delta!)
    X = np.vstack([genuine_vecs, stylistic_vecs])
    y = np.array([1] * len(genuine_vecs) + [0] * len(stylistic_vecs))

    # Нормализация
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr_scores = cross_val_score(lr, X_scaled, y, cv=5)
    print(f"\nLogistic Regression Accuracy: {lr_scores.mean():.3f} ± {lr_scores.std():.3f}")

    # MLP
    mlp = MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
    mlp_scores = cross_val_score(mlp, X_scaled, y, cv=5)
    print(f"MLP Accuracy: {mlp_scores.mean():.3f} ± {mlp_scores.std():.3f}")

    # Baseline
    baseline = max(np.mean(y), 1 - np.mean(y))
    print(f"Baseline: {baseline:.3f}")

    return lr_scores.mean(), mlp_scores.mean()

In [ ]:
# ============================================================
# ПОЛНЫЙ ИТОГОВЫЙ КОД: МНОГОСЛОЙНЫЙ АНАЛИЗ
# ============================================================

import os
import gc
import json
import re
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

# TransformerLens
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer

# ========== КОНФИГУРАЦИЯ ==========
CLASSIFIED_FILE = '/kaggle/input/datasets/mayasirotkina/dataset/classified_corrections.json'
SOURCE_FILE = '/kaggle/input/datasets/mayasirotkina/dataset/merged_results.json'
OUTPUT_DIR = '/kaggle/working/vectors_output_fixed'
MODEL_NAME = 'Qwen/Qwen3-8B'

LAYERS_TO_ANALYZE = [16, 20, 22, 24, 26, 28, 30, 32]
MAX_TOKENS = 256
MIN_GENUINE = 5
CONFIDENCE_FILTER = None
RANDOM_SEED = 42
# ==================================

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
Path(OUTPUT_DIR).mkdir(exist_ok=True)

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# ============================================================
# 1. ЗАГРУЗКА ДАННЫХ
# ============================================================
def load_classified(classified_file, source_file):
    print(f"Загружаю классификации: {classified_file}")
    with open(classified_file, 'r', encoding='utf-8') as f:
        classified = json.load(f)

    print(f"Загружаю оригинальные тексты: {source_file}")
    with open(source_file, 'r', encoding='utf-8') as f:
        source_data = json.load(f)

    source_lookup = {}
    for idx, q in enumerate(source_data.get('questions', [])):
        q_id = q.get('id', idx + 1)
        source_lookup[q_id] = {
            'question': q.get('question', ''),
            'full_thinking': q.get('full_thinking', ''),
        }

    genuine_cases, stylistic_cases = [], []

    for q in classified.get('questions', []):
        q_id = q['id']
        src = source_lookup.get(q_id, {})
        thinking = src.get('full_thinking', '')
        question = src.get('question', q.get('question_preview', ''))

        if not thinking:
            continue

        for marker_data in q.get('markers', []):
            marker = marker_data['marker']
            for cls_entry in marker_data.get('classifications', []):
                cls = cls_entry.get('classification', '')
                confidence = cls_entry.get('confidence', '')
                occ_idx = cls_entry.get('occurrence_index', 1)
                pos = cls_entry.get('position_in_tokens', 0)

                if CONFIDENCE_FILTER and confidence != CONFIDENCE_FILTER:
                    continue

                case = {
                    'question_id': q_id,
                    'question_text': question,
                    'thinking_text': thinking,
                    'marker': marker,
                    'occurrence_index': occ_idx,
                    'position_in_tokens': pos,
                    'classification': cls,
                    'confidence': confidence,
                }

                if cls == 'genuine_correction':
                    genuine_cases.append(case)
                elif cls == 'stylistic_filler':
                    stylistic_cases.append(case)

    print(f"\nGenuine: {len(genuine_cases)}")
    print(f"Stylistic: {len(stylistic_cases)}")
    return genuine_cases, stylistic_cases

# ============================================================
# 2. ПОСТРОЕНИЕ ПРОМПТОВ
# ============================================================
def build_prompts_for_case(case, tokenizer, max_tokens):
    thinking = case['thinking_text']
    marker = case['marker']

    tokens_list = re.findall(r'\w+|[^\w\s]', thinking.lower())
    occ_idx = case['occurrence_index'] - 1
    found_occ = 0
    actual_pos = None

    for i, tok in enumerate(tokens_list):
        if tok == marker.lower():
            if found_occ == occ_idx:
                actual_pos = i
                break
            found_occ += 1

    if actual_pos is None:
        return None, None

    text_before_marker = ' '.join(tokens_list[:actual_pos])
    text_with_marker = ' '.join(tokens_list[:actual_pos + 1])

    messages = [{"role": "user", "content": case['question_text']}]
    prefix = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    THINK_TOKEN = "<think>\n"
    correction_prompt = prefix + THINK_TOKEN + text_with_marker
    control_prompt = prefix + THINK_TOKEN + text_before_marker

    correction_tokens = tokenizer.encode(correction_prompt)[-max_tokens:]
    control_tokens = tokenizer.encode(control_prompt)[-max_tokens:]

    if len(control_tokens) < 5:
        return None, None

    return correction_tokens, control_tokens

# ============================================================
# 3. ИЗВЛЕЧЕНИЕ АКТИВАЦИЙ
# ============================================================
def extract_residual_stream(model, token_ids, layer, position=-1):
    input_tensor = torch.tensor([token_ids], dtype=torch.long)
    n_layers = model.cfg.n_layers
    actual_layer = layer if layer >= 0 else n_layers + layer

    hook_name = f'blocks.{actual_layer}.hook_resid_post'
    activations = {}

    def hook_fn(value, hook):
        activations['resid'] = value.detach().cpu().float().numpy()

    with torch.no_grad():
        model.run_with_hooks(
            input_tensor,
            fwd_hooks=[(hook_name, hook_fn)]
        )

    resid = activations['resid'][0, position, :]
    return resid

def compute_correction_vector(model, correction_tokens, control_tokens, layer):
    shared_pos = len(control_tokens) - 1
    act_correction = extract_residual_stream(model, correction_tokens, layer, position=shared_pos)
    act_control = extract_residual_stream(model, control_tokens, layer, position=-1)
    vector = act_control - act_correction
    return vector

# ============================================================
# 4. ИЗВЛЕЧЕНИЕ ВЕКТОРОВ ДЛЯ СЛОЯ
# ============================================================
def extract_vectors_for_layer(model, tokenizer, cases, label, layer, max_tokens):
    vectors = []
    meta = []
    skipped = 0

    print(f"\nИзвлекаю векторы [{label}] — {len(cases)} случаев...")

    for i, case in enumerate(cases):
        corr_tokens, ctrl_tokens = build_prompts_for_case(case, tokenizer, max_tokens)

        if corr_tokens is None:
            skipped += 1
            continue

        try:
            vec = compute_correction_vector(model, corr_tokens, ctrl_tokens, layer)
            vectors.append(vec)
            meta.append({
                'question_id': case['question_id'],
                'marker': case['marker'],
                'confidence': case['confidence'],
                'classification': case['classification'],
                'label': label,
            })
            if (i + 1) % 50 == 0:
                print(f"  [{i+1}/{len(cases)}] OK")
        except Exception as e:
            print(f"  Ошибка в примере {i}: {e}")
            skipped += 1

    print(f"  Извлечено: {len(vectors)}, пропущено: {skipped}")
    return np.vstack(vectors) if vectors else np.array([]), meta

# ============================================================
# 5. АНАЛИЗ СЛОЯ
# ============================================================
def analyze_layer(model, tokenizer, genuine_cases, stylistic_cases, layer, max_tokens):
    print(f"\n{'='*50}")
    print(f"ИЗВЛЕЧЕНИЕ ВЕКТОРОВ ДЛЯ СЛОЯ {layer}")
    print(f"{'='*50}")

    genuine_vecs, genuine_meta = extract_vectors_for_layer(
        model, tokenizer, genuine_cases, 'genuine', layer, max_tokens
    )
    stylistic_vecs, stylistic_meta = extract_vectors_for_layer(
        model, tokenizer, stylistic_cases, 'stylistic', layer, max_tokens
    )

    if len(genuine_vecs) == 0 or len(stylistic_vecs) == 0:
        print(f"Недостаточно данных для слоя {layer}, пропускаю")
        return None

    # Подготовка данных для классификации
    X = np.vstack([genuine_vecs, stylistic_vecs])
    y = np.array([1] * len(genuine_vecs) + [0] * len(stylistic_vecs))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Logistic Regression
    lr = LogisticRegression(max_iter=1000, random_state=RANDOM_SEED)
    lr_scores = cross_val_score(lr, X_scaled, y, cv=5)
    lr_mean, lr_std = lr_scores.mean(), lr_scores.std()

    # MLP
    mlp = MLPClassifier(
        hidden_layer_sizes=(100, 50),
        max_iter=1000,
        random_state=RANDOM_SEED,
        early_stopping=True
    )
    mlp_scores = cross_val_score(mlp, X_scaled, y, cv=5)
    mlp_mean, mlp_std = mlp_scores.mean(), mlp_scores.std()

    baseline = max(np.mean(y), 1 - np.mean(y))

    layer_result = {
        'layer': layer,
        'n_genuine': len(genuine_vecs),
        'n_stylistic': len(stylistic_vecs),
        'lr_accuracy': lr_mean,
        'lr_std': lr_std,
        'mlp_accuracy': mlp_mean,
        'mlp_std': mlp_std,
        'baseline': baseline,
        'improvement_over_baseline': lr_mean - baseline
    }

    print(f"\nРЕЗУЛЬТАТЫ ДЛЯ СЛОЯ {layer}:")
    print(f"  Logistic Regression: {lr_mean:.3f} ± {lr_std:.3f}")
    print(f"  MLP: {mlp_mean:.3f} ± {mlp_std:.3f}")
    print(f"  Baseline: {baseline:.3f}")
    print(f"  Улучшение над baseline: {lr_mean - baseline:+.3f}")

    # Сохраняем векторы
    np.savez(
        f'{OUTPUT_DIR}/vectors_layer_{layer}.npz',
        genuine_vectors=genuine_vecs,
        stylistic_vectors=stylistic_vecs
    )
    print(f"  Векторы сохранены: vectors_layer_{layer}.npz")

    return layer_result

# ============================================================
# 6. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# ============================================================
def plot_layer_results(results, output_dir):
    if not results:
        return

    results = sorted(results, key=lambda x: x['layer'])

    layers = [r['layer'] for r in results]
    lr_means = [r['lr_accuracy'] for r in results]
    lr_stds = [r['lr_std'] for r in results]
    mlp_means = [r['mlp_accuracy'] for r in results]
    mlp_stds = [r['mlp_std'] for r in results]
    baselines = [r['baseline'] for r in results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    x = np.arange(len(layers))
    width = 0.35

    ax1.errorbar(x - width/2, lr_means, yerr=lr_stds,
                fmt='o-', capsize=5, label='Logistic Regression',
                color='red', linewidth=2, markersize=8)
    ax1.errorbar(x + width/2, mlp_means, yerr=mlp_stds,
                fmt='s-', capsize=5, label='MLP',
                color='blue', linewidth=2, markersize=8)
    ax1.axhline(y=baselines[0], color='gray', linestyle='--',
                label='Baseline', alpha=0.7)

    ax1.set_xlabel('Layer')
    ax1.set_ylabel('Accuracy')
    ax1.set_title('Accuracy by Layer')
    ax1.set_xticks(x)
    ax1.set_xticklabels([str(l) for l in layers])
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0.45, 0.7])

    improvements = [r['improvement_over_baseline'] for r in results]
    colors = ['green' if imp > 0 else 'red' for imp in improvements]

    ax2.bar(layers, improvements, color=colors, alpha=0.7)
    ax2.axhline(y=0, color='black', linewidth=1)
    ax2.set_xlabel('Layer')
    ax2.set_ylabel('Improvement over baseline')
    ax2.set_title('Improvement by Layer')
    ax2.grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plot_path = f'{output_dir}/layer_analysis.png'
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"График сохранён: {plot_path}")

# ============================================================
# 7. СОХРАНЕНИЕ РЕЗУЛЬТАТОВ
# ============================================================
def save_results(results, output_dir):
    df = pd.DataFrame(results)
    csv_path = f'{output_dir}/layer_results.csv'
    df.to_csv(csv_path, index=False)
    print(f"Результаты сохранены в CSV: {csv_path}")

    json_path = f'{output_dir}/layer_results.json'
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump({
            'results': results,
            'metadata': {
                'model': MODEL_NAME,
                'max_tokens': MAX_TOKENS,
                'analysis_date': datetime.now().isoformat(),
                'best_layer': max(results, key=lambda x: x['lr_accuracy'])['layer'],
                'best_accuracy': max(results, key=lambda x: x['lr_accuracy'])['lr_accuracy']
            }
        }, f, ensure_ascii=False, indent=2)
    print(f"Результаты сохранены в JSON: {json_path}")

    print("\n" + "=" * 70)
    print("ИТОГОВАЯ СВОДКА ПО ВСЕМ СЛОЯМ")
    print("=" * 70)

    best_layer = max(results, key=lambda x: x['lr_accuracy'])
    print(f"\nЛучший слой: {best_layer['layer']}")
    print(f"   Accuracy: {best_layer['lr_accuracy']:.3f} ± {best_layer['lr_std']:.3f}")
    print(f"   Улучшение над baseline: {best_layer['improvement_over_baseline']:+.3f}")

    print("\nРезультаты по слоям:")
    print("-" * 70)
    for r in sorted(results, key=lambda x: x['layer']):
        print(f"  Слой {r['layer']:2d}: LR={r['lr_accuracy']:.3f}±{r['lr_std']:.3f} | "
              f"MLP={r['mlp_accuracy']:.3f}±{r['mlp_std']:.3f} | "
              f"n={r['n_genuine']}/{r['n_stylistic']}")

    return df

# ============================================================
# 8. MAIN
# ============================================================
print("=" * 70)
print("ЗАПУСК МНОГОСЛОЙНОГО АНАЛИЗА")
print("=" * 70)

# Загружаем данные
genuine_cases, stylistic_cases = load_classified(CLASSIFIED_FILE, SOURCE_FILE)

if len(genuine_cases) < MIN_GENUINE:
    print(f"\nОшибка: слишком мало genuine cases ({len(genuine_cases)} < {MIN_GENUINE})")
    raise SystemExit

# Очищаем память
clear_memory()

# Загружаем модель
print(f"\nЗагружаю модель {MODEL_NAME}...")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Устройство: {device}")
print(f"Доступно GPU: {torch.cuda.device_count()}")

model = HookedTransformer.from_pretrained(
    MODEL_NAME,
    device=None,
    n_devices=torch.cuda.device_count(),
    dtype=torch.float16,
    default_prepend_bos=False,
    fold_ln=False,
    center_writing_weights=False,
    move_to_device=True
)
model.eval()

hf_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(f"Модель загружена: {model.cfg.n_layers} слоёв, d_model={model.cfg.d_model}")
for i in range(torch.cuda.device_count()):
    print(f"Память GPU {i}: {torch.cuda.memory_allocated(i)/1e9:.2f} GB")

# Анализ по слоям
results = []
for layer in LAYERS_TO_ANALYZE:
    result = analyze_layer(
        model, hf_tokenizer,
        genuine_cases, stylistic_cases,
        layer, MAX_TOKENS
    )
    if result:
        results.append(result)
    clear_memory()

# Визуализация и сохранение
if results:
    plot_layer_results(results, OUTPUT_DIR)
    save_results(results, OUTPUT_DIR)

    print("\n" + "=" * 70)
    print("АНАЛИЗ ЗАВЕРШЕН")
    print("=" * 70)
    print(f"Результаты сохранены в папке: {OUTPUT_DIR}")

In [ ]:
!pip install transformer_lens